In [ ]:
!pip install -q sentence-transformers faiss-cpu pymupdf google-genai beautifulsoup4 requests


In [ ]:
import os
from google import genai
from getpass import getpass

api_key = getpass("Enter your Gemini API Key: ")

os.environ["GOOGLE_API_KEY"] = api_key

client = genai.Client(api_key=api_key)

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

print("Uploaded:", pdf_path)

In [ ]:
from google.colab import files
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print("Uploaded:", pdf_path)

In [ ]:
import fitz

def load_resume(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

resume_text = load_resume(pdf_path)
print("\nResume Loaded Successfully!")

In [ ]:
import re

def preprocess_resume(text):
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\+?\d[\d\s\-]{8,}', '', text)
    return text

resume_text = preprocess_resume(resume_text)

In [ ]:
def create_chunks(text):
    sections = re.split(r'\n+', text)
    chunks = []

    for sec in sections:
        sec = sec.strip()
        if len(sec) > 40:
            chunks.append(sec)

    return chunks

chunks = create_chunks(resume_text)
print("Total Chunks:", len(chunks))

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def embed_chunks(chunks):
    return embed_model.encode(chunks)

embeddings = embed_chunks(chunks)
print("Embeddings Created")

In [ ]:
import faiss
import numpy as np

def build_vector_store(embeddings):
    dimension = len(embeddings[0])
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings))
    return index

index = build_vector_store(embeddings)
print("Vector DB Ready")


In [ ]:
def query_vector_store(index, chunks, query, k=5):
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)
    return [chunks[i] for i in indices[0]]


In [ ]:
import requests
from bs4 import BeautifulSoup

def scrape_job_description(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text

In [ ]:
def extract_skills(text):
    prompt = f"""
    Extract ONLY technical skills from the text below.
    Return strictly as a Python list.

    Text:
    {text[:3000]}
    """
    return ask_gemini(prompt)

In [ ]:
def compare_skills(resume_skills, job_skills):
    prompt = f"""
    Compare these:

    Resume Skills:
    {resume_skills}

    Job Skills:
    {job_skills}

    Return ONLY:

    Matching Skills: []
    Missing Skills: []
    Match Percentage: %
    """
    return ask_gemini(prompt)

In [ ]:
def final_decision(result):
    prompt = f"""
    Based on:

    {result}

    If match percentage >= 70% → SELECTED
    Else → NOT SELECTED

    Output ONLY:
    Decision: SELECTED or NOT SELECTED
    """
    return ask_gemini(prompt)

In [ ]:
job_url = input("\nEnter Job URL: ")

print("\nScraping Job Description...")
job_text = scrape_job_description(job_url)


In [ ]:
relevant_chunks = query_vector_store(index, chunks, "skills experience technologies", k=5)
filtered_resume_text = " ".join(relevant_chunks)

In [ ]:
import google.generativeai as genai
import os

genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.5-flash')

def ask_gemini(prompt):
    response = model.generate_content(prompt)
    return response.text

print("\nExtracting Resume Skills...")
resume_skills = extract_skills(filtered_resume_text)
print("Resume Skills:", resume_skills)


In [ ]:
print("\nPlease paste the Job Description here (Press Enter twice to stop):")
job_lines = []
while True:
    line = input()
    if line == "":
        break
    job_lines.append(line)

job_text = "\n".join(job_lines)
print("\nJob Description captured.")

In [ ]:
print("\nExtracting Job Skills...")
job_skills = extract_skills(job_text)
print("Job Skills:", job_skills)

In [ ]:
print("\nComparing Skills...")
comparison = compare_skills(resume_skills, job_skills)
print("\n", comparison)

print("\nFinal Decision...")
decision = final_decision(comparison)
print(decision)